# **Dilatasyon, Erozyon ve Kenar Algılama**

####**Bu derste şunları öğreneceğiz:**
1. Dilatasyon (Genişletme)
2. Erozyon
3. Açılış
4. Kapanış
5. Canny Kenar Algılama

In [ ]:
import cv2
import numpy as np
from matplotlib import pyplot as plt

# Define our imshow function 
def imshow(title = "Image", image = None, size = 10):
    w, h = image.shape[0], image.shape[1]
    aspect_ratio = w/h
    plt.figure(figsize=(size * aspect_ratio,size))
    plt.imshow(cv2.cvtColor(image, cv2.COLOR_BGR2RGB))
    plt.title(title)
    plt.show()


- **Dilatasyon(Genişletme)** - Görüntüdeki nesnelerin sınırlarına piksel ekler
- **Erozyon** - Bir görüntüdeki nesnelerin sınırlarındaki pikselleri kaldırır
- **Açılma** - Erozyon ve ardından genişleme
- **Kapanış** - Dilatasyon ve ardından erozyon
Aşağıda 3x3 lük çekirdek ile yapılmış erozyon ve dilatasyon işlemi görülmektedir 

<img src="erosion.jpg" widht="400">

cv2.erode() şunu yapar:\
Kernel’in her bir konumunda merkezdeki pikselin değerini çekirdeğin kapladığı tüm bölgeye bakarak günceller.\
👉 Eğer çekirdeğin kapladığı alan tamamen beyaz (255) ise merkez piksel beyaz kalır.\
👉 Eğer çekirdeğin kapladığı alanda tek bir siyah (0) bile varsa, merkez piksel siyaha dönüşür.

cv2.dilate() ne yapar?\
Kernel yine görüntü üzerinde kaydırılır.\
Kernel’in merkezindeki pikselin yeni değeri şu kuralla belirlenir:\
👉 Eğer çekirdeğin kapladığı bölgede en az 1 tane beyaz (255) piksel varsa, merkez piksel beyaz yapılır.\
👉 Eğer çekirdeğin kapladığı bölgenin tamamı siyah (0) ise, merkez piksel siyah kalır.



In [ ]:
import cv2
import numpy as np

image = cv2.imread('../files/images/america10.jpg', 0)
imshow('Original', image)

# Çekirdek boyutumuzu tanımlayalım
kernel = np.ones((5,5), np.uint8)

# Şimdi aşındırıyoruz
erosion = cv2.erode(image, kernel, iterations = 1)
imshow('Erosion', erosion)

# Burayı genişletin
dilation = cv2.dilate(image, kernel, iterations = 1)
imshow('Dilation', dilation)

# Açma - Gürültüyü gidermek için 
opening = cv2.morphologyEx(image, cv2.MORPH_OPEN, kernel)
imshow('Opening',opening)

# Kapanış - Gürültüyü gidermek için 
closing = cv2.morphologyEx(image, cv2.MORPH_CLOSE, kernel)
imshow('Closing',closing)

## **Canny Edge Detection** 
<img src="deriv.jpg" width="600">\
### Kullanım:
edges = cv2.Canny(image, threshold1, threshold2)

Adım adım işlemler:\
Gürültü azaltma (Gaussian Blur).\
Gradyan (kenar yönü ve şiddeti) hesaplama.\
İki eşik değerine göre kenarları sınıflandırma (hysteresis thresholding).
Her pikselin gradyan büyüklüğü düşük ve yüksek eşiklerle kıyaslanır:

Gradyan < düşük eşik → kenar değil (siyah)\
Gradyan > yüksek eşik → kesin kenar (beyaz)\
Düşük < Gradyan < Yüksek → zayıf kenar (ancak güçlü bir kenara bağlıysa kenar kabul edilir).
### Parametreler:
1- image: Giriş görüntüsü (genelde grayscale yapılır). Renkli görüntüyü doğrudan verirsen, OpenCV otomatik griye çevirmeden işlem yapar ama sonuç istenen gibi olmayabilir → önce cv2.cvtColor(img, cv2.COLOR_BGR2GRAY) yapmak daha sağlıklı.\
2- threshold1 (alt eşik): Çift eşikleme (hysteresis) için düşük eşik değeri. Gradient değeri bu eşiğin altındaysa kenar sayılmaz.\
3- threshold2 (üst eşik): Çift eşikleme için yüksek eşik değeri. Gradient değeri bu eşiğin üzerindeyse kesin kenar kabul edilir. Eğer gradient bu iki eşik arasındaysa, ancak komşuluğunda güçlü kenar varsa kenar kabul edilir.\
Kenar algılama, hangi farkın/değişimin kenar olarak sayılması gerektiğini söylemek için bir eşiğe ihtiyaç duyar




In [ ]:
image = cv2.imread('../files/images/america11.jpg',0)

# Canny Kenar Algılama, gradyan değerlerini eşik olarak kullanır
# İlk eşik gradyanı
canny = cv2.Canny(image, 50, 120)
imshow('Canny 1', canny)
## İki değer sağlamamız gerekir: eşik1 ve eşik2. Eşik2'den büyük herhangi bir gradyan değeri
# bir kenar olarak kabul edilir. Eşik1'in altındaki herhangi bir değerin kenar olmadığı kabul edilir.
#Eşik1 ve eşik2 arasındaki değerler, değerlerinin nasıl olduğuna bağlı olarak 
# kenar ya da kenar olmayan olarak sınıflandırılır.
# yoğunluklar "bağlantılıdır". Bu durumda, 50'nin altındaki tüm gradyan değerleri 
#kenar olmayanlar olarak kabul edilir
#110'nin üzerindeki tüm değerler kenar olarak kabul edilir.

# Geniş kenar eşikleri çok sayıda kenar ekler
canny = cv2.Canny(image, 10, 200)
imshow('Canny Wide', canny)

# Dar eşik, daha az kenar ekler
canny = cv2.Canny(image, 200, 240)
imshow('Canny Narrow', canny)

canny = cv2.Canny(image, 60, 110)
imshow('Canny 4', canny)




#### **Auto Canny**

In [ ]:
def autoCanny(image):
  # Medyan görüntü piksel yoğunluğuna dayalı optimum eşikleri bulur
  blurred_img = cv2.blur(image, ksize=(5,5))
  med_val = np.median(image) 
  lower = int(max(0, 0.66 * med_val))
  upper = int(min(255, 1.33 * med_val))
  edges = cv2.Canny(image=image, threshold1=lower, threshold2=upper)
  return edges

auto_canny = autoCanny(image)
imshow("auto canny", auto_canny)

#Medyan değeri (np.median(image)) kullanıyor
#Görüntünün genel parlaklık yoğunluğunu ölçüyor.
#Çok karanlık veya çok aydınlık görüntülerde, uygun eşikler otomatik seçiliyor.
#Alt ve üst eşikler dinamik:
#lower ≈ 0.66 * median
#upper ≈ 1.33 * median
#Böylece görüntüye özel optimum eşikler elde ediliyor.
#Avantaj:
#Farklı ışık koşullarındaki görüntülerde sabit eşik yerine uyarlanabilir (adaptive) eşikler seçildiği için 
#daha güvenilir sonuç verir.
#Dezavantaj:
#Her görüntü için “mükemmel” sonuç vermez.